# K1 Cascade Inference — ResidualUNet2D (R → NS → B)
Notebook inference yang **memperbaiki dua bug** cascade lama:
1. **Renormalisasi antar-stage** — output R dinormalisasi ulang sebelum masuk NS, output NS sebelum U-Net (sesuai konvensi training: tiap model memetakan input-[0,1]-adaptif → target-[0,1]).
2. **Brightness saat simpan** — output disimpan di skala [0,1] → uint16 penuh, **tanpa** denormalisasi balik ke skala raw.

Juga: SSA memakai faktor konversi **1e-6** yang benar (m²/m³ → m²/cm³).

In [ ]:
!pip install -q tifffile scikit-image openpyxl pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports

In [ ]:
import os, gc, time, json, warnings, threading, multiprocessing
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from datetime import datetime
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import GradScaler, autocast

from tifffile import imread, imwrite
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_erosion
import psutil

warnings.filterwarnings('ignore')
CPU_COUNT = multiprocessing.cpu_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print('device:', device, '| CPU cores:', CPU_COUNT)

## 2. Config
**Verifikasi `PORE_CLASS_IDX`** harus sama dengan training U-Net segmentasi.

In [ ]:
class Cfg:
    GDRIVE_ROOT = '/content/drive/MyDrive/Dataset TA'

    # ─── Checkpoint blok R & NS: Residual U-Net 2D (restorasi) ─────────
    CKPT_R  = '/content/drive/MyDrive/Alfian_TA/K1_blokR_resunet2d/resunet2d_R_best.pth'
    CKPT_NS = '/content/drive/MyDrive/Alfian_TA/K1_blokNS_resunet2d/resunet2d_NS_best.pth'
    # ─── Checkpoint blok B: Residual U-Net 2.5D segmentasi ─────────────
    UNET_CKPT = f'{GDRIVE_ROOT}/Res-Net/best_seg_model.pth'

    # ─── Data input (RAW ber-ring yang akan di-cascade) ───────────────
    FILE_RAW   = f'{GDRIVE_ROOT}/317/Bentheimer 317/Bentheimer_grayscale.tif'
    FILE_GT    = f'{GDRIVE_ROOT}/317/Bentheimer 317/Bentheimer_binary.tif'      # mask biner GT (pore>0). None bila tak ada.
    FILE_CLEAN = f'{GDRIVE_ROOT}/317/Bentheimer 317/Bentheimer_clean_lowrot.tif' # GT grayscale bersih. None bila tak ada.

    SLICE_START  = None
    SLICE_END    = None
    SPATIAL_CROP = None

    # ─── Restorasi (R/NS) — IDENTIK dgn training ResUNet2D ─────────────
    PATCH_SIZE = 256
    STRIDE     = 96
    USE_AMP    = False         # samakan dgn training (FP32)

    # ─── Segmentasi U-Net 2.5D (blok B) ───────────────────────────────
    UNET_THRESH    = 0.5
    UNET_IN_CH     = 5
    PORE_CLASS_IDX = 0         # <-- VERIFIKASI: harus sama dgn training U-Net seg

    # ─── Petrofisika ──────────────────────────────────────────────────
    VOXEL_SIZE_UM   = 5.714
    KC_TORTUOSITY   = 2.5
    KC_SHAPE_FACTOR = 2.0

    SAVE_DIR = '/content/drive/MyDrive/Alfian_TA/K1_cascade_resunet2d_output'

cfg = Cfg()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)

print("="*68); print("  K1 CASCADE (ResUNet2D) — cek file"); print("="*68)
for label, path in [("R    ",cfg.CKPT_R),("NS   ",cfg.CKPT_NS),("B    ",cfg.UNET_CKPT),
                    ("RAW  ",cfg.FILE_RAW),("GTbin",cfg.FILE_GT or '(—)'),
                    ("GTgry",cfg.FILE_CLEAN or '(—)')]:
    st = ("OK" if os.path.isfile(path) else "TIDAK DITEMUKAN") if path and path!='(—)' else "—"
    print(f"  {label}: {st}  {path}")
print("  Output:", cfg.SAVE_DIR)

## 3. Model restorasi: ResidualUNet2D

In [ ]:
# ============================================================================
#  Residual U-Net 2D — varian RESTORASI (regresi grayscale->grayscale)
#  Adaptasi dari Wang et al. (2024, SPE J) Residual U-Net segmentasi:
#    - output 1 channel (bukan softmax n_classes)
#    - GLOBAL RESIDUAL: out = net(x) + x  (model belajar residual noise/ring)
#    - FILTERS [32,64,128,256,512] -> receptive field besar utk ring artifact
# ============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                       nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = ResidualBlock(in_ch, out_ch)
        self.pool  = nn.MaxPool2d(2, 2)
    def forward(self, x):
        skip = self.block(x)
        return self.pool(skip), skip

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.block = ResidualBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.block(torch.cat([x, skip], dim=1))

class ResidualUNet2D(nn.Module):
    # Residual U-Net 2D restorasi. in_ch=1. global residual: final = net(x)+x
    FILTERS = [32, 64, 128, 256, 512]
    def __init__(self, in_ch=1, out_ch=1, global_residual=True):
        super().__init__()
        f = self.FILTERS
        self.global_residual = global_residual
        self.enc1 = EncoderBlock(in_ch, f[0]); self.enc2 = EncoderBlock(f[0], f[1])
        self.enc3 = EncoderBlock(f[1], f[2]);  self.enc4 = EncoderBlock(f[2], f[3])
        self.bottleneck = ResidualBlock(f[3], f[4])
        self.dec4 = DecoderBlock(f[4], f[3], f[3]); self.dec3 = DecoderBlock(f[3], f[2], f[2])
        self.dec2 = DecoderBlock(f[2], f[1], f[1]); self.dec1 = DecoderBlock(f[1], f[0], f[0])
        self.out  = nn.Conv2d(f[0], out_ch, kernel_size=1)
    def forward(self, x):
        inp = x
        d, s1 = self.enc1(x); d, s2 = self.enc2(d)
        d, s3 = self.enc3(d); d, s4 = self.enc4(d)
        d = self.bottleneck(d)
        d = self.dec4(d, s4); d = self.dec3(d, s3)
        d = self.dec2(d, s2); d = self.dec1(d, s1)
        r = self.out(d)
        # global residual: prediksi residual lalu tambah input (in_ch=1 → x langsung)
        y = r + inp if self.global_residual else r
        return y

def count_parameters(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 4. Load checkpoint R & NS

In [ ]:
def load_resunet2d(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = ResidualUNet2D(in_ch=1, out_ch=1, global_residual=True).to(device)
    state = ckpt['model_state'] if 'model_state' in ckpt else ckpt
    m.load_state_dict(state); m.eval()
    arch = ckpt.get('arch','?'); ep = ckpt.get('epoch','?')
    print(f"  ResUNet2D loaded: {os.path.basename(ckpt_path)} | arch={arch} epoch={ep}")
    return m

model_R  = load_resunet2d(cfg.CKPT_R,  device)
model_NS = load_resunet2d(cfg.CKPT_NS, device)

## 5. Data helpers (load + normalize_volume)

In [ ]:
def load_and_crop_volume(tif_path, crop_size=None, crop_origin=None):
    print(f"  Loading {os.path.basename(tif_path)} ...", end='', flush=True)
    vol = imread(tif_path).astype(np.float32); print(f" shape={vol.shape}")
    if crop_size is not None:
        D,H,W = vol.shape; cs = crop_size
        if crop_origin is not None: x0,y0,z0 = crop_origin
        else: x0=max(0,(W-cs)//2); y0=max(0,(H-cs)//2); z0=max(0,(D-cs)//2)
        x0=min(x0,W-cs); y0=min(y0,H-cs); z0=min(z0,D-cs)
        vol = vol[z0:z0+cs, y0:y0+cs, x0:x0+cs]
    return vol

def normalize_volume(vol, p1=None, p99=None):
    # Normalisasi adaptif p1/p99; return (vol01, p1, p99) utk denorm balik
    if p1  is None: p1  = np.percentile(vol, 1)
    if p99 is None: p99 = np.percentile(vol, 99)
    vn = np.clip((vol - p1)/(p99 - p1 + 1e-8), 0.0, 1.0)
    return vn.astype(np.float32), float(p1), float(p99)

def split_slices(n, tf, vf):
    nt = int(n*tf); nv = int(n*vf)
    return list(range(0,nt)), list(range(nt,nt+nv)), list(range(nt+nv,n))

class PatchDataset(Dataset):
    def __init__(self, noisy_vol, clean_vol, slice_indices, patch_size,
                 patches_per_slice=20, augment=False, mode='train'):
        self.noisy=noisy_vol; self.clean=clean_vol; self.patch=patch_size
        self.augment=augment; self.mode=mode; self.patches_per_slice=patches_per_slice
        if mode=='train':
            self.slice_indices=slice_indices
            self.length=len(slice_indices)*patches_per_slice
        else:
            stride=max(patch_size//2,32); self.coords=[]; _,H,W=noisy_vol.shape
            for z in slice_indices:
                for y in range(0,H-patch_size+1,stride):
                    for x in range(0,W-patch_size+1,stride):
                        self.coords.append((z,y,x))
            self.length=len(self.coords)
    def __len__(self): return self.length
    def _augment(self,n,c):
        k=np.random.randint(0,4); n=np.rot90(n,k); c=np.rot90(c,k)
        if np.random.rand()>0.5: n=np.fliplr(n); c=np.fliplr(c)
        if np.random.rand()>0.5: n=np.flipud(n); c=np.flipud(c)
        return n.copy(),c.copy()
    def __getitem__(self,idx):
        ps=self.patch; _,H,W=self.noisy.shape
        if self.mode=='train':
            slice_idx=self.slice_indices[idx//self.patches_per_slice]
            y=np.random.randint(0,H-ps); x=np.random.randint(0,W-ps)
        else:
            z,y,x=self.coords[idx]; slice_idx=z
        n_p=self.noisy[slice_idx,y:y+ps,x:x+ps]; c_p=self.clean[slice_idx,y:y+ps,x:x+ps]
        if self.augment and self.mode=='train': n_p,c_p=self._augment(n_p,c_p)
        return torch.from_numpy(n_p[None]).float(), torch.from_numpy(c_p[None]).float()

## 6. Blok B: U-Net 2.5D segmentasi + load

In [ ]:
# ============================================================================
#  Blok B: Residual U-Net 2.5D SEGMENTASI (arsitektur identik notebook cascade lama)
#  CATATAN: ini SEGMENTASI (softmax, n_classes=2), beda dari ResUNet2D restorasi.
# ============================================================================
class _SegResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1=nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False); self.bn1=nn.BatchNorm2d(out_ch)
        self.conv2=nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False); self.bn2=nn.BatchNorm2d(out_ch)
        self.relu=nn.ReLU(inplace=True)
        self.shortcut=(nn.Sequential(nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch))
                       if in_ch!=out_ch else nn.Identity())
    def forward(self,x):
        idt=self.shortcut(x); o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o))
        return self.relu(o+idt)
class _SegEnc(nn.Module):
    def __init__(self,i,o): super().__init__(); self.block=_SegResidualBlock(i,o); self.pool=nn.MaxPool2d(2,2)
    def forward(self,x): s=self.block(x); return self.pool(s),s
class _SegDec(nn.Module):
    def __init__(self,i,sk,o):
        super().__init__(); self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True)
        self.block=_SegResidualBlock(i+sk,o)
    def forward(self,x,s):
        x=self.up(x)
        if x.shape[2:]!=s.shape[2:]: x=F.interpolate(x,size=s.shape[2:],mode='bilinear',align_corners=True)
        return self.block(torch.cat([x,s],1))
class ResidualUNetSeg(nn.Module):
    FILTERS=[32,64,128,256,512]
    def __init__(self,in_ch=5,n_classes=2):
        super().__init__(); f=self.FILTERS
        self.enc1=_SegEnc(in_ch,f[0]); self.enc2=_SegEnc(f[0],f[1])
        self.enc3=_SegEnc(f[1],f[2]);  self.enc4=_SegEnc(f[2],f[3])
        self.bottleneck=_SegResidualBlock(f[3],f[4])
        self.dec4=_SegDec(f[4],f[3],f[3]); self.dec3=_SegDec(f[3],f[2],f[2])
        self.dec2=_SegDec(f[2],f[1],f[1]); self.dec1=_SegDec(f[1],f[0],f[0])
        self.out=nn.Conv2d(f[0],n_classes,1)
    def forward(self,x):
        x,s1=self.enc1(x); x,s2=self.enc2(x); x,s3=self.enc3(x); x,s4=self.enc4(x)
        x=self.bottleneck(x)
        x=self.dec4(x,s4); x=self.dec3(x,s3); x=self.dec2(x,s2); x=self.dec1(x,s1)
        return self.out(x)

def load_unet_seg(ckpt_path, device):
    ckpt=torch.load(ckpt_path, map_location=device, weights_only=False)
    m=ResidualUNetSeg(in_ch=cfg.UNET_IN_CH, n_classes=2)
    state=ckpt['state_dict'] if 'state_dict' in ckpt else (ckpt['model_state'] if 'model_state' in ckpt else ckpt)
    m.load_state_dict(state); m.to(device).eval()
    print(f"  U-Net seg loaded | epoch={ckpt.get('epoch','?')}")
    return m

unet = load_unet_seg(cfg.UNET_CKPT, device)

## 7. Inference helpers (patch+Hann)

In [ ]:
@torch.no_grad()
def infer_slice(model, sl2d, dev, ps=256, stride=96, use_amp=False):
    # Inference 1 slice 2D, patch + Hann blending, output skala [0,1]
    H,W=sl2d.shape; out=np.zeros((H,W),np.float64); wgt=np.zeros((H,W),np.float64)
    g=np.outer(np.hanning(ps),np.hanning(ps))+1e-6
    ys=list(range(0,H-ps+1,stride)); xs=list(range(0,W-ps+1,stride))
    if not ys or ys[-1]+ps<H: ys.append(max(0,H-ps))
    if not xs or xs[-1]+ps<W: xs.append(max(0,W-ps))
    model.eval()
    for y in ys:
        for x in xs:
            t=torch.from_numpy(sl2d[y:y+ps,x:x+ps][None,None]).float().to(dev)
            with autocast('cuda', enabled=use_amp):
                p=model(t).squeeze().float().cpu().numpy()
            out[y:y+ps,x:x+ps]+=p*g; wgt[y:y+ps,x:x+ps]+=g
    return np.clip(out/wgt,0,1).astype(np.float32)

def infer_volume(model, vol01, dev, ps, stride, use_amp, tag='infer'):
    o=np.zeros_like(vol01)
    for z in tqdm(range(vol01.shape[0]), desc=tag):
        o[z]=infer_slice(model, vol01[z], dev, ps, stride, use_amp)
        if z%50==0 and z>0:
            gc.collect()
            if dev.type=='cuda': torch.cuda.empty_cache()
    return o

## 8. Load RAW (+ GT opsional)

In [ ]:
# ===== LOAD RAW + normalisasi adaptif (p1/p99 input sendiri) =====
print("="*68); print("  LOAD RAW"); print("="*68)
raw = load_and_crop_volume(cfg.FILE_RAW, cfg.SPATIAL_CROP, None)
if cfg.SLICE_START is not None or cfg.SLICE_END is not None:
    s = cfg.SLICE_START or 0; e = cfg.SLICE_END if cfg.SLICE_END is not None else raw.shape[0]
    raw = raw[s:e]

vol_raw, raw_p1, raw_p99 = normalize_volume(raw)
n_slices = vol_raw.shape[0]
print(f"  RAW: {vol_raw.shape} range_asli=[{raw.min():.1f},{raw.max():.1f}] p1={raw_p1:.2f} p99={raw_p99:.2f}")
del raw; gc.collect()

# mask GT biner (Dice/IoU)
mask_gt=None
if cfg.FILE_GT and os.path.isfile(cfg.FILE_GT):
    mg=load_and_crop_volume(cfg.FILE_GT, cfg.SPATIAL_CROP, None)
    if cfg.SLICE_START is not None or cfg.SLICE_END is not None:
        s=cfg.SLICE_START or 0; e=cfg.SLICE_END if cfg.SLICE_END is not None else mg.shape[0]; mg=mg[s:e]
    mask_gt=(mg>0.5).astype(np.uint8); print(f"  MASK GT: {mask_gt.shape}"); del mg; gc.collect()

# GT grayscale bersih (PSNR/SSIM) — normalisasi adaptif sendiri
vol_clean=None; has_gt_gray=False
if cfg.FILE_CLEAN and os.path.isfile(cfg.FILE_CLEAN):
    cln=load_and_crop_volume(cfg.FILE_CLEAN, cfg.SPATIAL_CROP, None)
    if cfg.SLICE_START is not None or cfg.SLICE_END is not None:
        s=cfg.SLICE_START or 0; e=cfg.SLICE_END if cfg.SLICE_END is not None else cln.shape[0]; cln=cln[s:e]
    vol_clean,_,_=normalize_volume(cln); has_gt_gray=True
    assert vol_clean.shape==vol_raw.shape
    print(f"  GT grayscale: {vol_clean.shape}"); del cln; gc.collect()
else:
    print("  GT grayscale: (tidak ada) — PSNR/SSIM dilewati")

## 9. Segmentasi U-Net 2.5D

In [ ]:
@torch.no_grad()
def segment_unet(vol01, model, device, threshold=0.5):
    n=vol01.shape[0]; masks=np.zeros_like(vol01, dtype=np.uint8)
    for i in tqdm(range(n), desc='U-Net seg', unit='slice'):
        stack=np.stack([vol01[max(0,min(i+off,n-1))] for off in (-2,-1,0,1,2)],axis=0)
        t=torch.from_numpy(stack).float().unsqueeze(0).to(device)
        prob=F.softmax(model(t),dim=1)[0,cfg.PORE_CLASS_IDX].cpu().numpy()
        masks[i]=(prob>=threshold).astype(np.uint8)
    return masks

## 10. Fungsi petrofisika (SSA fix 1e-6)

In [ ]:
VOX_M = cfg.VOXEL_SIZE_UM*1e-6
def porosity(b): return {'total_frac':float(b.mean()),'per_slice':b.mean(axis=(1,2))}
def ssa_total(b, vox_m):
    surf=b.astype(int)-binary_erosion(b).astype(int)
    D,H,W=b.shape; sa=surf.sum()*vox_m**2; vol=D*H*W*vox_m**3
    Sv=sa/vol if vol>0 else 0.0
    return {'SSA_m2_cm3':float(Sv*1e-6),'Sv_per_m':float(Sv),'surface_voxels':int(surf.sum())}  # FIX: 1e-6 (1 m3=1e6 cm3)
def ssa_per_slice(b, vox_m):
    Sv=np.zeros(b.shape[0])
    for z in range(b.shape[0]):
        sl=b[z]; surf=sl.astype(int)-binary_erosion(sl).astype(int)
        H,W=sl.shape; vol=H*W*vox_m**3
        Sv[z]=surf.sum()*vox_m**2/vol if vol>0 else 0.0
    return Sv
def kozeny_carman(phi, Sv, tau=2.5, k0=2.0):
    if phi<=0 or phi>=1 or Sv<=0: return float('nan'),float('nan')
    K=phi**3/(k0*tau**2*Sv**2*(1-phi)**2); return float(K),float(K/9.869233e-16)
def dice_iou(pred, ref):
    p,r=pred.astype(bool),ref.astype(bool)
    tp=np.logical_and(p,r).sum(); fp=np.logical_and(p,~r).sum(); fn=np.logical_and(~p,r).sum()
    return float(2*tp/(2*tp+fp+fn+1e-8)), float(tp/(tp+fp+fn+1e-8))
print("petrofisika siap (SSA pakai faktor 1e-6 yang benar)")

## 11. CASCADE (FIX renormalisasi antar-stage)

In [ ]:
# ============================================================================
#  CASCADE R -> NS -> B  (FIX: renormalisasi adaptif di TIAP batas tahap)
#  Konvensi training: tiap model memetakan input-[0,1]-adaptif -> target-[0,1].
#  Maka output R harus DI-normalisasi ulang sebelum masuk NS, dst.
# ============================================================================
t0=time.time()

print("\n[1/3] Blok R (deringing) ...")
vol_dering = infer_volume(model_R, vol_raw, device, cfg.PATCH_SIZE, cfg.STRIDE, cfg.USE_AMP, "R: dering")

print("\n[2/3] Blok NS (denoising) ...")
vol_dering_n, _, _ = normalize_volume(vol_dering)         # <-- FIX renormalisasi sebelum NS
vol_denoise = infer_volume(model_NS, vol_dering_n, device, cfg.PATCH_SIZE, cfg.STRIDE, cfg.USE_AMP, "NS: denoise")

print("\n[3/3] Blok B (segmentasi U-Net 2.5D) ...")
vol_denoise_n, _, _ = normalize_volume(vol_denoise)       # <-- FIX renormalisasi sebelum U-Net
mask_pred = segment_unet(vol_denoise_n, unet, device, cfg.UNET_THRESH)

dt=time.time()-t0; pf=mask_pred.mean()
print(f"\n  Cascade selesai {dt/60:.1f} mnt | pore_fraction K1 = {pf:.3f}")
if pf>0.65: print("    [warn] pore fraction >0.65 — cek polaritas/PORE_CLASS_IDX vs training U-Net.")

## 12. Metrik grayscale (vs GT)

In [ ]:
df_signal=None
if has_gt_gray:
    print("="*68); print("  METRIK GRAYSCALE per-slice (vs GT bersih)"); print("="*68)
    from scipy.stats import pearsonr
    def _m(c,x):
        return (psnr_fn(c,x,data_range=1.0), ssim_fn(c,x,data_range=1.0),
                float(np.sqrt(np.mean((c-x)**2))), float(np.mean(np.abs(c-x))),
                float(np.mean(x-c)), float(pearsonr(c.ravel(),x.ravel())[0]))
    rec=[]
    for z in tqdm(range(n_slices), desc='signal'):
        c=vol_clean[z]
        pr=_m(c,vol_raw[z]); pdg=_m(c,vol_dering[z]); pn=_m(c,vol_denoise[z])
        rec.append({'slice':z,
            'PSNR_raw':pr[0],'PSNR_dering':pdg[0],'PSNR_denoise':pn[0],
            'SSIM_raw':pr[1],'SSIM_dering':pdg[1],'SSIM_denoise':pn[1],
            'RMSE_raw':pr[2],'RMSE_dering':pdg[2],'RMSE_denoise':pn[2],
            'Corr_raw':pr[5],'Corr_dering':pdg[5],'Corr_denoise':pn[5]})
    df_signal=pd.DataFrame(rec)
    _s=lambda c:f"{df_signal[c].mean():.4f}±{df_signal[c].std():.4f}"
    print(f"\n  {'Metrik':<6}{'RAW':>20}{'R(dering)':>20}{'NS(denoise)':>20}")
    for m in ['PSNR','SSIM','RMSE','Corr']:
        print(f"  {m:<6}{_s(m+'_raw'):>20}{_s(m+'_dering'):>20}{_s(m+'_denoise'):>20}")
    df_signal.to_csv(os.path.join(cfg.SAVE_DIR,'K1_metrics_per_slice.csv'),index=False)
    print("  tersimpan: K1_metrics_per_slice.csv")
else:
    print("  GT grayscale tidak ada — metrik dilewati.")

## 13. Petrofisika + vs GT

In [ ]:
por_pred=porosity(mask_pred); ssa_pred=ssa_total(mask_pred,VOX_M)
phi=por_pred['total_frac']
Km2,KmD=kozeny_carman(phi,ssa_pred['Sv_per_m'],cfg.KC_TORTUOSITY,cfg.KC_SHAPE_FACTOR)
print("="*60); print("  PETROFISIKA PREDIKSI K1"); print("="*60)
print(f"  Porositas: {phi*100:.3f} %")
print(f"  SSA      : {ssa_pred['SSA_m2_cm3']:.6f} m2/cm3 (Sv={ssa_pred['Sv_per_m']:.1f}/m)")
print(f"  K (KC)   : {KmD:.4e} mD")

rows=[]
if mask_gt is not None:
    assert mask_gt.shape==mask_pred.shape
    dice,iou=dice_iou(mask_pred,mask_gt)
    por_gt=porosity(mask_gt); ssa_gt=ssa_total(mask_gt,VOX_M); phi_gt=por_gt['total_frac']
    _,KmD_gt=kozeny_carman(phi_gt,ssa_gt['Sv_per_m'],cfg.KC_TORTUOSITY,cfg.KC_SHAPE_FACTOR)
    print("="*60); print("  K1 vs GROUND TRUTH"); print("="*60)
    print(f"  Dice {dice:.4f} | IoU {iou:.4f}")
    print(f"  Porositas GT/pred: {phi_gt*100:.3f}/{phi*100:.3f} % (|galat| {abs(phi-phi_gt)*100:.3f})")
    print(f"  SSA GT/pred: {ssa_gt['SSA_m2_cm3']:.6f}/{ssa_pred['SSA_m2_cm3']:.6f} m2/cm3")
    print(f"  K GT/pred: {KmD_gt:.4e}/{KmD:.4e} mD")
    rows.append({'skema':'K1','Dice':dice,'IoU':iou,
        'porositas_GT_%':phi_gt*100,'porositas_pred_%':phi*100,
        'SSA_GT_m2cm3':ssa_gt['SSA_m2_cm3'],'SSA_pred_m2cm3':ssa_pred['SSA_m2_cm3'],
        'K_GT_mD':KmD_gt,'K_pred_mD':KmD})
else:
    rows.append({'skema':'K1','porositas_pred_%':phi*100,
        'SSA_pred_m2cm3':ssa_pred['SSA_m2_cm3'],'K_pred_mD':KmD})
df=pd.DataFrame(rows); df.to_csv(os.path.join(cfg.SAVE_DIR,'K1_metrics.csv'),index=False)
print("  tersimpan: K1_metrics.csv"); df

## 14. Panel visual

In [ ]:
mid=n_slices//2
panels=[('1. RAW',vol_raw[mid])]
if has_gt_gray: panels.append(('GT grayscale',vol_clean[mid]))
panels+=[('2. Deringed (R)',vol_dering[mid]),('3. Denoised (NS)',vol_denoise[mid]),('4. Mask K1 (B)',mask_pred[mid])]
if mask_gt is not None: panels.append(('Mask GT',mask_gt[mid]))
nc=len(panels); fig,ax=plt.subplots(1,nc,figsize=(4.2*nc,5))
for a,(t,im) in zip(ax,panels): a.imshow(im,cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(cfg.SAVE_DIR,'cascade_panel.png'),dpi=140); plt.show()

## 15. Simpan (FIX brightness)

In [ ]:
# ===== SIMPAN (TANPA bug brightness: skala [0,1] -> u16 penuh, tanpa denorm ke raw) =====
print("="*68); print("  SIMPAN GRAYSCALE + MASK"); print("="*68)
print(f"  Range dering : [{vol_dering.min():.3f},{vol_dering.max():.3f}]")
print(f"  Range denoise: [{vol_denoise.min():.3f},{vol_denoise.max():.3f}]  (skala [0,1] sama)")

imwrite(os.path.join(cfg.SAVE_DIR,'K1_1_deringed_u16.tif'), (np.clip(vol_dering,0,1)*65535).astype(np.uint16))
imwrite(os.path.join(cfg.SAVE_DIR,'K1_2_denoised_u16.tif'), (np.clip(vol_denoise,0,1)*65535).astype(np.uint16))
imwrite(os.path.join(cfg.SAVE_DIR,'K1_1_deringed_float.tif'), vol_dering.astype(np.float32))
imwrite(os.path.join(cfg.SAVE_DIR,'K1_2_denoised_float.tif'), vol_denoise.astype(np.float32))
imwrite(os.path.join(cfg.SAVE_DIR,'K1_3_mask.tif'), (mask_pred*255).astype(np.uint8))
print("  ✓ deringed/denoised (u16 + float), mask (pore=255)")
print("\n  Di Fiji: set display range SEMUA jendela ke 0–65535 utk perbandingan brightness jujur.")
print("  Tersimpan di", cfg.SAVE_DIR)